# 14 — ログと症状からの診断

動画の印象ではなく、因果鎖のどこが先に崩れたかを時系列で見つけます。

**前提**: `13_end_to_end_baseline.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
# 背景: ログ診断を上流PyMPCと同じリポジトリ配置・環境変数で再現します。
# 目的: ワークスペースとPyMPCの場所を確定し、後続セルの診断条件を再現可能にします。
# OSに依存しないパス演算を行うためPathを読み込む。
from pathlib import Path
# 環境変数の設定にos、モジュール検索パスの設定にsysを使う。
import os, sys

# Notebookを起動した現在位置を絶対パスへ正規化する。
ROOT = Path.cwd().resolve()
# notebook_pympc直下から起動した場合だけリポジトリルートへ移る。
if ROOT.name == "notebook_pympc":
    # 外部実装をROOT基準で参照できるよう親ディレクトリを採用する。
    ROOT = ROOT.parent
# 上流Quadruped-PyMPCの配置先をROOTから組み立てる。
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
# 誤った起動場所のログを診断しないよう実装の存在を検証する。
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
# 同名モジュールの取り違えを防ぐため検索パス未登録時だけ処理する。
if str(PYMPC_ROOT) not in sys.path:
    # 現行リポジトリの実装を最優先でimportするため先頭へ追加する。
    sys.path.insert(0, str(PYMPC_ROOT))

# acados生成物の探索基準を未設定時だけ上流同梱ディレクトリへ合わせる。
os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
# 画面のない診断環境でもMuJoCoを描画できるよう未設定時はEGLを選ぶ。
os.environ.setdefault("MUJOCO_GL", "egl")
# 診断対象のワークスペースを目視確認できるよう表示する。
print("workspace :", ROOT)
# 上流実装の参照先を目視確認できるよう表示する。
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## 最小ログ

- reference / measured: COM位置、速度、roll/pitch/yaw
- gait: phase、current contact、予測contact
- foothold: 離地、参照着地、実着地
- MPC: GRF、予測状態、status、solve time、cost
- low level: Jacobian、raw torque、clip後torque、飽和flag
- Plant: 実接触、実GRF、滑り速度

目標GRFと実GRFを同じ名前で保存しないことが重要です。

In [2]:
# 背景: 症状の因果順を調べるには、同じ時刻軸上の速度・姿勢・トルクから定量指標を作ります。
# 目的: 4 sの学習用ログを生成し、vx RMSE・roll RMS・トルク飽和率を計算します。
# 時系列生成、乱数、三角関数、統計量の計算にNumPyを使う。
import numpy as np
# 再実行時に同じ速度ノイズを得るためseed=0の乱数生成器を作る。
rng = np.random.default_rng(0)
# 0〜4 sをdt=0.01 s相当の401点で表す時刻配列、shape (401,)を作る。
t = np.linspace(0, 4, 401)
# world前進速度の参照値を全時刻0.3 m/sとする同shape配列にする。
ref_vx = np.full_like(t, 0.3)
# 一次遅れ応答0.3(1-exp(-2t))へ標準偏差0.02 m/sの観測ノイズを加える。
vx = 0.3*(1-np.exp(-2*t)) + 0.02*rng.normal(size=t.size)
# rollを振幅0.03 rad、歩容周波数1.35 Hzの正弦波として生成する。
roll = 0.03*np.sin(2*np.pi*1.35*t)
# 関節トルクを基準18 N mと正半波8 N mの周期負荷として生成する。
torque = 18 + 8*np.maximum(0, np.sin(2*np.pi*1.35*t))
# 90% soft limitに対応する飽和閾値を21.33 N mへ固定する。
limit = 21.33

# 同じ4 s時間窓から追従・姿勢・アクチュエータ指標をまとめる。
metrics = {
    # world前進速度誤差vx-ref_vxの二乗平均平方根[m/s]を計算する。
    "vx_rmse": np.sqrt(np.mean((vx-ref_vx)**2)),
    # roll角の二乗平均平方根[rad]を計算する。
    "roll_rms_rad": np.sqrt(np.mean(roll**2)),
    # torque≥21.33 N mとなるサンプルの割合[0,1]を計算する。
    "torque_saturation_rate": np.mean(torque >= limit),
}
# 3つの指標辞書をNotebookのexecute_resultとして表示する。
metrics

{'vx_rmse': np.float64(0.07794550867930047),
 'roll_rms_rad': np.float64(0.02134342006883563),
 'torque_saturation_rate': np.float64(0.39650872817955113)}

## 診断順

1. solver status/NaN
2. contact予測と実接触のずれ
3. 摩擦marginとGRFの急変
4. torque飽和
5. 姿勢・速度誤差
6. 転倒

最後に見えた転倒ではなく、最初に閾値を越えた内部量を原因候補にします。

In [3]:
# 背景: 診断では動画の印象ではなく、事前に定めた単位付き閾値を同じ時間窓の指標へ適用します。
# 目的: 速度・roll・トルク飽和の各指標を対応閾値と比較し、OK/NGを一覧表示します。
# vx RMSE[m/s]、roll RMS[rad]、飽和率[0,1]の許容上限を名前対応で定義する。
thresholds = {"vx_rmse": 0.12, "roll_rms_rad": 0.08, "torque_saturation_rate": 0.05}
# 計算済み3指標を保存順に走査して各閾値と比較する。
for key, value in metrics.items():
    # 値が対応上限を超えればNG、それ以外はOKとして小数4桁で表示する。
    print(f"{key:26s} {value:.4f}  {'NG' if value > thresholds[key] else 'OK'}")

vx_rmse                    0.0779  OK
roll_rms_rad               0.0213  OK
torque_saturation_rate     0.3965  NG


## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。